# Lesson 03 — SQLAlchemy ORM Models

**Run this in Google Colab** — every student gets the same environment.

---

## Step 1: Install dependencies

In [1]:
!pip install sqlalchemy oracledb alembic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.2 MB/s eta 0:00:00


## Step 2: Define ORM models (Team, User, Task, Comment)

In [9]:
from sqlalchemy import (
    create_engine, Column, Integer, String,
    Text, ForeignKey, DateTime, func
)
from sqlalchemy.orm import declarative_base, relationship, Session

Base = declarative_base()

class Team(Base):
    __tablename__ = 'teams'
    id          = Column(Integer, primary_key=True)
    name        = Column(String(50), nullable=False, unique=True)
    description = Column(String(200))
    created_at  = Column(DateTime, server_default=func.current_timestamp())
    users       = relationship('User', back_populates='team')

    def __repr__(self):
        return f"<Team(id={self.id}, name='{self.name}')>"


class User(Base):
    __tablename__ = 'users'
    id         = Column(Integer, primary_key=True)
    username   = Column(String(50), nullable=False, unique=True)
    email      = Column(String(100), nullable=False)
    full_name  = Column(String(100))
    team_id    = Column(Integer, ForeignKey('teams.id'))
    created_at = Column(DateTime, server_default=func.current_timestamp())
    team       = relationship('Team', back_populates='users')
    tasks      = relationship('Task', back_populates='assignee')

    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"


class Task(Base):
    __tablename__ = 'tasks'
    id          = Column(Integer, primary_key=True)
    title       = Column(String(200), nullable=False)
    description = Column(String(1000))
    status      = Column(String(20), default='open')
    assigned_to = Column(Integer, ForeignKey('users.id'))
    created_at  = Column(DateTime, server_default=func.current_timestamp())
    updated_at  = Column(DateTime, onupdate=func.current_timestamp())
    assignee    = relationship('User', back_populates='tasks')
    comments    = relationship('Comment', back_populates='task', cascade='all, delete-orphan')

    def __repr__(self):
        return f"<Task(id={self.id}, title='{self.title}', status='{self.status}')>"

class Comment(Base):
    __tablename__ = 'comments'
    id         = Column(Integer, primary_key=True)
    task_id    = Column(Integer, ForeignKey('tasks.id', ondelete='CASCADE'), nullable=False)
    user_id    = Column(Integer, ForeignKey('users.id'), nullable=False)
    content    = Column(Text, nullable=False)
    created_at = Column(DateTime, server_default=func.current_timestamp())
    task       = relationship('Task', back_populates='comments')
    user       = relationship('User')

    def __repr__(self):
        return f"<Comment(id={self.id}, task_id={self.task_id})>"


print('✅ Models defined: Team, User, Task, Comment')

✅ Models defined: Team, User, Task, Comment


## Step 3: Connect to DB and query with ORM

In [10]:
# ============================================================
# CONFIGURATION — replace with your FreeSQL credentials
# ============================================================
USERNAME = 'A01644644_SCHEMA_TN9BO'
PASSWORD = '3!KPEMHAF8Y9nPRXFGCZUQAAP3EDM5'
DSN      = 'tcps://db.freesql.com:2484/26ai_un3c1'

engine = create_engine(
    'oracle+oracledb://:@',
    connect_args={
        'user': USERNAME,
        'password': PASSWORD,
        'dsn': DSN
    }
)

with Session(engine) as session:
    print('🏢 Teams:')
    for team in session.query(Team).all():
        print(f'   {team}')
        for user in team.users:
            print(f'      -> {user.full_name} ({user.username})')

    print('\n📝 Tasks with assignees:')
    for task in session.query(Task).all():
        assignee = task.assignee.full_name if task.assignee else 'Unassigned'
        print(f'   {task.title} → {assignee}')

print('\n✅ ORM models working! Relationships navigate automatically.')

🏢 Teams:
   <Team(id=1, name='Engineering')>
      -> Alice Smith (alice_dev)
      -> Bob Jones (bob_dev)
   <Team(id=2, name='Product')>
      -> Carol White (carol_pm)

📝 Tasks with assignees:
   Fix login bug → Alice Smith
   Design new dashboard → Carol White
   Update dependencies → Bob Jones

✅ ORM models working! Relationships navigate automatically.


---
# Alembic Migrations
## Step 4: Initialize Alembic

In [11]:
import os
from alembic.config import Config
from alembic import command

alembic_cfg = Config()
alembic_cfg.set_main_option('script_location', '/content/project/alembic')
alembic_cfg.set_main_option('sqlalchemy.url', 'oracle+oracledb://:@')

!mkdir -p /content/project/alembic/versions

env_py = f'''
from sqlalchemy import engine_from_config
from alembic import context
from __main__ import Base

config = context.config
target_metadata = Base.metadata

def run_migrations_online():
    connectable = engine_from_config(
        config.get_section(config.config_ini_section),
        prefix="sqlalchemy.",
        connect_args={{"user": "{USERNAME}", "password": "{PASSWORD}", "dsn": "{DSN}"}}
    )
    with connectable.connect() as connection:
        context.configure(connection=connection, target_metadata=target_metadata)
        with context.begin_transaction():
            context.run_migrations()

run_migrations_online()
'''

with open('/content/project/alembic/env.py', 'w') as f:
    f.write(env_py)

script_template = '''"""${message}

Revision ID: ${up_revision}
Revises: ${down_revision | comma,n}
Create Date: ${create_date}
"""

from alembic import op
import sqlalchemy as sa
${imports if imports else ""}

revision = ${repr(up_revision)}
down_revision = ${repr(down_revision)}
branch_labels = ${repr(branch_labels)}
depends_on = ${repr(depends_on)}

def upgrade():
${upgrades if upgrades else "    pass"}

def downgrade():
${downgrades if downgrades else "    pass"}
'''

with open('/content/project/alembic/script.py.mako', 'w') as f:
    f.write(script_template)

print('✅ Alembic initialized in /content/project/alembic/')

✅ Alembic initialized in /content/project/alembic/


## Exercise 2 — Generate migration for Comment table

In [12]:
command.revision(alembic_cfg, autogenerate=True, message='add comments table')

import glob
migration_files = sorted(glob.glob('/content/project/alembic/versions/*.py'))
for f in migration_files:
    print(f)

latest = migration_files[-1]
with open(latest) as f:
    print(f.read())

Generating /content/project/alembic/versions/a87bca8ff4dd_add_comments_table.py ...  done
/content/project/alembic/versions/a87bca8ff4dd_add_comments_table.py
"""add comments table

Revision ID: a87bca8ff4dd
Revises: 
Create Date: 2026-05-12 14:21:06.471658
"""

from alembic import op
import sqlalchemy as sa
from sqlalchemy.dialects import oracle

revision = 'a87bca8ff4dd'
down_revision = None
branch_labels = None
depends_on = None

def upgrade():
# ### commands auto generated by Alembic - please adjust! ###
    op.create_table('comments',
    sa.Column('id', sa.Integer(), nullable=False),
    sa.Column('task_id', sa.Integer(), nullable=False),
    sa.Column('user_id', sa.Integer(), nullable=False),
    sa.Column('content', sa.Text(), nullable=False),
    sa.Column('created_at', sa.DateTime(), server_default=sa.text('CURRENT_TIMESTAMP'), nullable=True),
    sa.ForeignKeyConstraint(['task_id'], ['tasks.id'], ondelete='CASCADE'),
    sa.ForeignKeyConstraint(['user_id'], ['users.id'], ),


## Step 5: Apply migration

In [13]:
command.upgrade(alembic_cfg, 'head')
print('Migration applied')

✅ Migration applied!


---
## Exercise 3 — CRUD Challenge

In [15]:
with Session(engine) as session:
    team = Team(name='DevOps')
    session.add(team)
    session.flush()

    user = User(
        username='diana_ops',
        email='diana@devops.com',
        full_name='Diana Ops',
        team_id=team.id
    )
    session.add(user)
    session.flush()

    tasks = [
      Task(title='Setup CI pipeline', assigned_to=user.id),
      Task(title='Monitor alerts',    assigned_to=user.id),
      Task(title='Update docs',       assigned_to=user.id),
    ]
    session.add_all(tasks)
    session.flush()

    count = session.query(Task).filter_by(assigned_to=user.id).count()
    print(f'Task count: {count}')

    tasks[0].status = 'closed'
    print(f'Closed: {tasks[0].title}')

    session.delete(tasks[2])
    print(f'Deleted: {tasks[2].title}')

    session.commit()
    print('Done')

Task count: 3
Closed: Setup CI pipeline
Deleted: Update docs
Done


---
## Exercise 4 — Migration Rollback

In [16]:
command.downgrade(alembic_cfg, '-1')

---
## Cleanup — Delete generated migration files

In [ ]:
import glob
import os

migration_files = glob.glob('/content/project/alembic/versions/*.py')
for f in migration_files:
    os.remove(f)
    print(f'Deleted: {f}')